In [1]:
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [2]:
!pip install pytesseract
!apt-get install tesseract-ocr -y
!pip install pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Surya resume.pdf to Surya resume.pdf
Saving Harsha Resume.pdf to Harsha Resume.pdf
Saving Resume_Rammanoshankarfinal.pdf to Resume_Rammanoshankarfinal.pdf


In [4]:
from pypdf import PdfReader

documents = []

for file_name in uploaded.keys():
    pdf = PdfReader(file_name)

    text = ""

    for page in pdf.pages:
        text += page.extract_text()

    documents.append({
        "resume_name": file_name,
        "content": text
    })

In [5]:
print("Number of resumes:", len(documents))

for doc in documents:
    print(doc["resume_name"])

Number of resumes: 3
Surya resume.pdf
Harsha Resume.pdf
Resume_Rammanoshankarfinal.pdf


In [6]:
def chunk_text(text, chunk_size=500, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [7]:
chunks = []

for doc in documents:

    split_text = chunk_text(doc["content"])

    for chunk in split_text:

        chunks.append({
            "resume_name": doc["resume_name"],
            "text": chunk
        })

print("Total Chunks:", len(chunks))

Total Chunks: 13


In [8]:
!pip install -q sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
print(embeddings.shape)

(13, 384)


In [11]:
!pip install -q chromadb

In [19]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="resume_collection"
)

In [20]:
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(texts))],
    metadatas=[
        {"resume": chunk["resume_name"]}
        for chunk in chunks
    ]
)

In [21]:
collection.count()

13

In [22]:
query = "Which candidate is best suited for a Data Scientist role?"

In [23]:
query_embedding = embedding_model.encode(
    query
).tolist()

In [26]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

In [27]:
print(results["documents"][0])

['                             \n     Percentage: 96.8                                                                   Percentage: 94.8  \nCareer Objective \nEnthusiastic and disciplined B.Tech Artificial Intelligence & Data Science student with a strong foundation in \nprogramming, databases, and machine learning concepts. Seeking opportunities to apply technical skills in real-world \nprojects, internships, or entry-level roles while continuously learning and contributing to organizational growth. ', 'nships, or entry-level roles while continuously learning and contributing to organizational growth. \nSkills \nLanguages: \nAdvance:  Python | Java | HTML | CSS | SQL | JavaScript  Core subjects: Artificial Intelligence | Machine Learning | \nComputer Vision  Tools: VS Code, Git, Jupyter Notebook,Tableau,Docker \n \nProjects \nAI Job Recommendation System using NCO Codes \n• Designed and developed an AI-based system that analyzes student details and recommends suitable job roles \nusi

In [28]:
from groq import Groq

client = Groq(
    api_key="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"
)

retrieved_chunks = results["documents"][0]

context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

In [29]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Based on the provided context, the candidate best suited for a Data Scientist role is RAMMANOSHANKAR P. 

This is because the candidate has:

- A strong foundation in programming languages such as Python, Java, and SQL.
- Experience in machine learning concepts and has worked on projects involving AI-based applications such as object detection and recommendation systems.
- Hands-on experience in developing ML and DL models using various tools and technologies such as TensorFlow, Scikit-Learn, Keras, and PyTorch.
- A certification in Python for Data Science from NPTEL, IIT Madras.
- Experience in data analysis and has worked on projects that involve analyzing student details and recommending suitable job roles using National Classification of Occupations (NCO) codes.

These skills and experiences make the candidate a strong fit for a Data Scientist role.


In [30]:
prompt = f"""
You are an HR recruiter.

Based ONLY on the resume information provided below,
identify which candidate is best suited for a Generative AI Engineer role.

Explain:
1. Candidate name
2. Relevant skills
3. Relevant projects
4. Certifications
5. Why the candidate is the best fit

Context:
{context}

Question:
{query}
"""

In [31]:
for doc in documents:
    print(doc["resume_name"])
    print(len(doc["content"]))
    print("-"*50)

Surya resume.pdf
0
--------------------------------------------------
Harsha Resume.pdf
2632
--------------------------------------------------
Resume_Rammanoshankarfinal.pdf
2033
--------------------------------------------------
